# Quantisation-Aware Training (QAT)

Purpose: Implements QAT manually using a fake-quantisation function with a straight-through estimator (STE), rather than TFMOT.

Approach:
1. Rebuild the baseline architecture using custom `QATConv2D` / `QATDense` layers that fake-quantise weights in the forward pass, with STE gradients in the backward pass.
2. Also fake-quantise activations after each ReLU, using a per-batch dynamic range with STE. This simulates the precision loss a real int8 model would see at inference time.
3. Fine-tune from the FP32 baseline's pretrained weights for the faster convergence and standard practice.
4. After fine-tuning, convert to a real int8 `.tflite` model via the standard `TFLiteConverter`. The fake-quant training is what makes the weights robust to this final conversion step.

In [ ]:
# import
import tensorflow as tf
import numpy as np
import os
import time
import csv

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/tinyml-quant-security'

In [ ]:
# Load CIFAR-10 and reapply the same fixed split as in other notebooks
(x_train_full, y_train_full), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
x_train_full = x_train_full.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
y_train_full = y_train_full.flatten()
y_test = y_test.flatten()

split = np.load(f'{PROJECT_DIR}/results/data_split.npz')
train_idx, val_idx = split['train_idx'], split['val_idx']

x_train, y_train = x_train_full[train_idx], y_train_full[train_idx]
x_val, y_val = x_train_full[val_idx], y_train_full[val_idx]

print(f"Train: {x_train.shape}, Val: {x_val.shape}, Test: {x_test.shape}")

In [ ]:
# Same tf.data augmentation pipeline as the baseline notebook
BATCH_SIZE = 64

def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.resize_with_crop_or_pad(image, 36, 36)
    image = tf.image.random_crop(image, size=[32, 32, 3])
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.shuffle(len(x_train), seed=SEED).map(augment, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

## Fake-quantisation with straight-through estimator (STE)

- Forward pass: simulate int8 rounding
- Backward pass: gradients pass through unchanged. This is what makes the rounding operation trainable instead of being non-differentiable.

In [ ]:
def make_fake_quantize(num_bits=8):
    qmax = float(2 ** (num_bits - 1) - 1)

    @tf.custom_gradient
    def fq(x):
        scale = tf.maximum(tf.reduce_max(tf.abs(x)), 1e-8) / qmax
        x_q = tf.round(x / scale) * scale

        def grad(dy):
            # Straight-through estimator = pass gradient through unchanged
            return dy

        return x_q, grad

    return fq

fake_quantize_weights = make_fake_quantize(num_bits=8)
fake_quantize_activations = make_fake_quantize(num_bits=8)

In [ ]:
class QATConv2D(tf.keras.layers.Conv2D):
    """Conv2D with fake-quantised weights
    (forward: rounded, backward: STE)"""
    def call(self, inputs):
        q_kernel = fake_quantize_weights(self.kernel)
        outputs = tf.nn.conv2d(
            inputs, q_kernel, strides=[1, *self.strides, 1],
            padding=self.padding.upper()
        )
        if self.use_bias:
            outputs = tf.nn.bias_add(outputs, self.bias)
        if self.activation is not None:
            outputs = self.activation(outputs)
        return outputs


class QATDense(tf.keras.layers.Dense):
    """Dense with fake-quantised weights
    (forward: rounded, backward: STE)"""
    def call(self, inputs):
        q_kernel = fake_quantize_weights(self.kernel)
        outputs = tf.matmul(inputs, q_kernel)
        if self.use_bias:
            outputs = tf.nn.bias_add(outputs, self.bias)
        if self.activation is not None:
            outputs = self.activation(outputs)
        return outputs


class FakeQuantActivation(tf.keras.layers.Layer):
    """Fake-quantises activations after a layer
    it's like simulating int8 activation precision loss)"""
    def call(self, inputs):
        return fake_quantize_activations(inputs)

In [ ]:
# QAT model architecture (same structure as baseline one and with fake-quant layers)
def build_qat_model(num_classes=10):
    inputs = tf.keras.layers.Input(shape=(32, 32, 3))

    x = QATConv2D(32, 3, padding='same', activation='relu', name='conv1')(inputs)
    x = tf.keras.layers.BatchNormalization(name='bn1')(x)
    x = FakeQuantActivation()(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)

    x = QATConv2D(64, 3, padding='same', activation='relu', name='conv2')(x)
    x = tf.keras.layers.BatchNormalization(name='bn2')(x)
    x = FakeQuantActivation()(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)

    x = QATConv2D(128, 3, padding='same', activation='relu', name='conv3')(x)
    x = tf.keras.layers.BatchNormalization(name='bn3')(x)
    x = FakeQuantActivation()(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)

    x = tf.keras.layers.Flatten()(x)
    x = QATDense(128, activation='relu', name='dense1')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = QATDense(num_classes, activation='softmax', name='dense2')(x)

    return tf.keras.Model(inputs, outputs)

qat_model = build_qat_model()
qat_model.summary()

## Load pretrained baseline weights

- Fine-tuning from the FP32 baseline (rather than training from scratch because it is converges much faster).
- Layer names match the baseline architecture from baseline_training notebook, so weights transfer directly.

In [ ]:
baseline_model = tf.keras.models.load_model(f'{PROJECT_DIR}/models/baseline.keras')

# Transfer weights layer-by-layer where shapes match (Conv2D/Dense/BatchNorm)
# QATConv2D/QATDense subclass Conv2D/Dense so weight shapes are identical
baseline_weight_layers = [l for l in baseline_model.layers if l.weights]
qat_weight_layers = [l for l in qat_model.layers if l.weights]

assert len(baseline_weight_layers) == len(qat_weight_layers), (
    f"Layer count mismatch: baseline has {len(baseline_weight_layers)} weighted layers, "
    f"QAT model has {len(qat_weight_layers)}. Check both architectures match."
)

for src, dst in zip(baseline_weight_layers, qat_weight_layers):
    dst.set_weights(src.get_weights())

print("Transferred baseline weights into QAT model.")

In [ ]:
# to check the accuracy first
qat_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

pre_finetune_loss, pre_finetune_acc = qat_model.evaluate(x_test, y_test)
print(f"QAT model BEFORE fine-tuning - Test accuracy: {pre_finetune_acc:.4f}")

In [ ]:
# Fine-tune with fake-quantisation
# Using lower learning rate than baseline training because the model is fine-tuned from the baseline model
QAT_EPOCHS = 15 # the model can convergence quickly because it starts from a good weight

qat_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=3, min_lr=1e-6),
]

qat_history = qat_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=QAT_EPOCHS,
    callbacks=qat_callbacks
)

In [ ]:
post_finetune_loss, post_finetune_acc = qat_model.evaluate(x_test, y_test)
print(f"QAT model AFTER fine-tuning - Test accuracy: {post_finetune_acc:.4f}")

In [ ]:
# Save the fake-quant trained model
qat_model.save(f'{PROJECT_DIR}/models/qat_model.keras')

In [ ]:
# Convert the fine-tuned QAT model to int8 TFLite model
def build_plain_model(num_classes=10):
    inputs = tf.keras.layers.Input(shape=(32, 32, 3))
    x = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu', name='conv1')(inputs)
    x = tf.keras.layers.BatchNormalization(name='bn1')(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)

    x = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu', name='conv2')(x)
    x = tf.keras.layers.BatchNormalization(name='bn2')(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)

    x = tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu', name='conv3')(x)
    x = tf.keras.layers.BatchNormalization(name='bn3')(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)

    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128, activation='relu', name='dense1')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', name='dense2')(x)
    return tf.keras.Model(inputs, outputs)

plain_qat_model = build_plain_model()

qat_weight_layers2 = [l for l in qat_model.layers if l.weights]
plain_weight_layers = [l for l in plain_qat_model.layers if l.weights]
for src, dst in zip(qat_weight_layers2, plain_weight_layers):
    dst.set_weights(src.get_weights())

# Check the accuracy which should be closed to post_finetune_acc above
plain_qat_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
_, plain_check_acc = plain_qat_model.evaluate(x_test, y_test)
print(f"Plain (no fake-quant ops) QAT model - Test accuracy: {plain_check_acc:.4f}")
print(f"Should be closely matched with fake-quant model accuracy: {post_finetune_acc:.4f}")

plain_qat_model.export(f'{PROJECT_DIR}/models/qat_savedmodel')

In [ ]:
# Standard int8 TFLite conversion using the same procedure as the PTQ notebook
NUM_CALIBRATION_SAMPLES = 300
rng = np.random.RandomState(SEED)
calib_indices = rng.choice(len(x_train), NUM_CALIBRATION_SAMPLES, replace=False)
calib_data = x_train[calib_indices]

def representative_dataset():
    for i in range(len(calib_data)):
        sample = calib_data[i:i+1].astype(np.float32)
        yield [sample]

converter = tf.lite.TFLiteConverter.from_saved_model(f'{PROJECT_DIR}/models/qat_savedmodel')
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

qat_tflite_model = converter.convert()

qat_model_path = f'{PROJECT_DIR}/models/qat_model.tflite'
with open(qat_model_path, 'wb') as f:
    f.write(qat_tflite_model)

print(f"Saved QAT TFLite model to {qat_model_path}")
print(f"QAT model size: {os.path.getsize(qat_model_path) / 1024:.2f} KB")

## Clean evaluation (same way as PTQ notebook)

In [ ]:
interpreter = tf.lite.Interpreter(model_path=qat_model_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]
input_scale, input_zero_point = input_details['quantization']
print(f"Input quantisation - scale: {input_scale}, zero_point: {input_zero_point}")


def quantize_input(x_float, scale, zero_point):
    x_int8 = x_float / scale + zero_point
    return np.clip(np.round(x_int8), -128, 127).astype(np.int8)


def evaluate_tflite_model(interpreter, x_test, y_test, input_scale, input_zero_point, num_samples=None):
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    if num_samples is None:
        num_samples = len(x_test)

    correct = 0
    latencies = []

    for i in range(num_samples):
        x_sample = quantize_input(x_test[i:i+1], input_scale, input_zero_point)

        start = time.perf_counter()
        interpreter.set_tensor(input_details['index'], x_sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details['index'])
        latencies.append(time.perf_counter() - start)

        pred = np.argmax(output[0])
        if pred == y_test[i]:
            correct += 1

    accuracy = correct / num_samples
    avg_latency_ms = np.mean(latencies) * 1000
    p95_latency_ms = np.percentile(latencies, 95) * 1000

    return accuracy, avg_latency_ms, p95_latency_ms


qat_accuracy, qat_avg_latency, qat_p95_latency = evaluate_tflite_model(
    interpreter, x_test, y_test, input_scale, input_zero_point
)
qat_size_kb = os.path.getsize(qat_model_path) / 1024

print(f"QAT - Accuracy: {qat_accuracy:.4f}")
print(f"QAT - Avg latency: {qat_avg_latency:.3f} ms, P95 latency: {qat_p95_latency:.3f} ms")
print(f"QAT - Model size: {qat_size_kb:.2f} KB")

In [ ]:
# Append to shared results CSV
results_path = f'{PROJECT_DIR}/results/clean_eval.csv'
file_exists = os.path.isfile(results_path)

with open(results_path, 'a', newline='') as f:
    writer = csv.writer(f)
    if not file_exists:
        writer.writerow(['variant', 'accuracy', 'avg_latency_ms', 'p95_latency_ms', 'size_kb'])
    writer.writerow(['QAT', qat_accuracy, qat_avg_latency, qat_p95_latency, qat_size_kb])

print(f"Appended QAT results to {results_path}")